# DUBAI TOWER — Floor 03: Semantic Room Graph + GNN Room-Type Prediction

Semantic room-graph + GNN room-type prediction for the Dubai Tower **Level 03**
apartments, using the Modified Swiss Dwellings (MSD) graph-ML node schema:

1. Load each **room type** OBJ as closed **cells** (rooms = graph nodes, typed).
2. Connect rooms through the **doors** (`aperture.obj`) — a door in the wall between two
   rooms creates a **circulation edge** (matched by proximity, since real walls have
   thickness so room volumes don't share faces).
3. Build the **semantic room-circulation graph** and visualise it.
4. Export the dataset in the **MSD CSV schema** (`nodes/edges/graphs.csv`).
5. Train a **GraphSAGE node classifier** (PyTorch-Geometric, via `topologicpy.PyG`) to
   **predict each room's type** from its features + graph neighbourhood, then evaluate and
   visualise true vs predicted.

> No pretrained MSD model is required — we train our own classifier on this graph
> (transductive node classification with train/val/test node masks).

## 1. Imports

In [1]:
import os, time, random
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.PyG import PyG
from collections import Counter

random.seed(42); np.random.seed(42)
renderer = "vscode"

## 2. Configuration — paths, room types, MSD feature mappings

In [2]:
from pathlib import Path

# OBJ folder (rooms, doors, windows for Floor 03)
FLOOR_TAG = "F03"
def _find_obj_dir(tag):
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "assets" / "obj" / tag
        if (p / "bedroom.obj").exists():
            return p
    raise FileNotFoundError(f"Could not find assets/obj/{tag} with bedroom.obj")
OBJ_DIR = _find_obj_dir(FLOOR_TAG)
DATASET_DIR = Path.cwd() / "Exports" / "Prediction_F03"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
print("OBJ_DIR    :", OBJ_DIR)
print("DATASET_DIR:", DATASET_DIR)

# Room-type OBJ files present for Floor 03 (filename = semantic label)
ROOM_FILES = {
    "bedroom":    "bedroom.obj",
    "livingroom": "livingroom.obj",
    "kitchen":    "kitchen.obj",
    "corridor":   "corridor.obj",
    "stairs":     "stairs.obj",
    "bathroom":   "bathroom.obj",
    "storeroom":  "storeroom.obj",
    "balcony":    "balcony.obj",
}
DOOR_FILE   = "aperture.obj"   # interior doors -> circulation edges
WINDOW_FILE = "window.obj"     # exterior openings -> per-room feature

# --- MSD label + feature mappings -------------------------------------------
# Reference MSD class ids (9 classes). We remap to CONTIGUOUS ids for the classes
# actually present (PyG needs labels 0..K-1 with no gaps).
ROOM_LABEL = {"bedroom":0,"livingroom":1,"kitchen":2,"dining":3,"corridor":4,
              "stairs":5,"storeroom":6,"bathroom":7,"balcony":8}
# Node feature: zoning group (private / social / service / outdoor)
ZONING = {"bedroom":[1,0,0,0],"livingroom":[0,1,0,0],"kitchen":[0,1,0,0],"dining":[0,1,0,0],
          "corridor":[0,1,0,0],"stairs":[0,0,1,0],"storeroom":[0,0,1,0],"bathroom":[0,0,1,0],
          "balcony":[0,0,0,1]}
# Node feature: connectivity role (circulation vs enclosed)
NODE_CONN = {"bedroom":[0,1,0],"livingroom":[0,1,0],"kitchen":[0,1,0],"dining":[0,1,0],
             "corridor":[1,0,0],"stairs":[1,0,0],"storeroom":[0,1,0],"bathroom":[0,1,0],
             "balcony":[0,1,0]}
# Edge feature: door connectivity (we treat every aperture as a "door")
DOOR_CONN = [0,1,0]
# Display colours per room type
ROOM_COLOR = {"bedroom":"#FFBFBF","bathroom":"#4444FF","corridor":"#7FFFBF","kitchen":"#BF3F3F",
              "livingroom":"#FFBF00","stairs":"#BF3FFF","storeroom":"#FF7FFF","dining":"#A0522D",
              "balcony":"#007F00","unknown":"#AAAAAA"}

PROXIMITY_TOL = 0.6   # max distance (m) from a door/window centroid to a room cell to count as touching

OBJ_DIR    : c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F03
DATASET_DIR: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03


## 3. Load room OBJs as closed cells (the graph nodes)

Each room type OBJ is imported with `transposeAxes=True` (so plan is in X-Y, height in Z) and
each room object is rebuilt into a watertight **Cell** via `Cell.ByFaces` at progressive
tolerances. The room type (from the filename) is stored on every cell.

In [3]:
def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

cells, rtypes = [], []
for rtype, fn in ROOM_FILES.items():
    path = str(OBJ_DIR / fn)
    if not os.path.exists(path):
        print(f"  [SKIP] missing {fn}"); continue
    objs = Topology.ByOBJPath(path, transposeAxes=True)
    if not isinstance(objs, list): objs = [objs]
    n = 0
    for obj in objs:
        if obj is None: continue
        faces = Topology.Faces(obj) or []
        if len(faces) < 4: continue
        c = build_cell(faces)
        if c is None: continue
        d = Dictionary.ByKeysValues(["room_type", "color"], [rtype, ROOM_COLOR.get(rtype, "#AAAAAA")])
        c = Topology.SetDictionary(c, d)
        cells.append(c); rtypes.append(rtype); n += 1
    print(f"  {rtype:11s}: {n} cells")

N = len(cells)
centroids = [Topology.Centroid(c) for c in cells]
cen_xyz = np.array([[Vertex.X(v), Vertex.Y(v), Vertex.Z(v)] for v in centroids])
print(f"\nTotal rooms (nodes): {N}")
print("Distribution:", dict(Counter(rtypes)))

# Contiguous local labels for the classes actually present (PyG needs 0..K-1, no gaps)
present  = sorted(set(rtypes), key=lambda t: ROOM_LABEL[t])
LOCAL    = {t: i for i, t in enumerate(present)}
NAME     = {i: t for t, i in LOCAL.items()}
N_CLASS  = len(present)
print("Local labels:", LOCAL)

  bedroom    : 20 cells
  livingroom : 9 cells
  kitchen    : 9 cells
  corridor   : 15 cells
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
  stairs     : 3 cells
  bathroom   : 26 cells
  storeroom  : 10 cells
  balcony 

## 4. Visualise the room cells coloured by type

In [4]:
fig = Topology.Show(cells, faceColorKey="color", faceOpacity=0.85,
                    showEdges=True, edgeColor="black", edgeWidth=1,
                    showVertices=False, backgroundColor="white",
                    width=1100, height=750, showFigure=False, renderer=renderer)
fig.update_layout(
    title=dict(text="Floor 03 — rooms coloured by type", font=dict(color="black")),
    paper_bgcolor="white",
    scene=dict(aspectmode="data",
               xaxis=dict(color="black"), yaxis=dict(color="black"), zaxis=dict(color="black")),
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=0, y=0, z=2.2)))
fig.show(renderer=renderer)

# colour legend
print("Room-type colours:")
for t in present:
    print(f"  {t:11s} {ROOM_COLOR[t]}")

Room-type colours:
  bedroom     #FFBFBF
  livingroom  #FFBF00
  kitchen     #BF3F3F
  corridor    #7FFFBF
  stairs      #BF3FFF
  storeroom   #FF7FFF
  bathroom    #4444FF
  balcony     #007F00


## 5. Load doors + windows, and build the circulation edges

Doors (`aperture.obj`) and windows are flat faces. Real walls have thickness, so room volumes
don't share faces — instead each **door** is matched by **proximity** to the two nearest room
cells; those two rooms get a circulation edge. Each **window** is matched to its single nearest
room (a per-room feature).

In [5]:
def load_faces(fn):
    path = str(OBJ_DIR / fn)
    if not os.path.exists(path):
        print(f"  [SKIP] missing {fn}"); return []
    objs = Topology.ByOBJPath(path, selfMerge=False)
    if not isinstance(objs, list): objs = [objs]
    faces = []
    for obj in objs:
        if obj is None: continue
        fs = Topology.Faces(obj) or []
        if fs:
            faces.extend(fs)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is None:
                    w2 = Topology.RemoveCollinearEdges(w); f = Face.ByWire(w2) if w2 else None
                if f is not None: faces.append(f)
    return faces

doors   = load_faces(DOOR_FILE)
windows = load_faces(WINDOW_FILE)
print(f"doors: {len(doors)}   windows: {len(windows)}")

def nearest_rooms(face, k, maxd):
    fc = Topology.Centroid(face)
    ds = sorted((Vertex.Distance(fc, cells[i]), i) for i in range(N))
    return [i for d, i in ds if d <= maxd][:k]

# Door -> edge between its 2 nearest rooms
edge_set = set()
unmatched_doors = 0
for d in doors:
    near = nearest_rooms(d, 2, PROXIMITY_TOL)
    if len(near) >= 2:
        a, b = sorted(near[:2])
        if a != b: edge_set.add((a, b))
    else:
        unmatched_doors += 1
edges = sorted(edge_set)

# Window -> nearest room (feature)
win_count = [0] * N
for w in windows:
    near = nearest_rooms(w, 1, PROXIMITY_TOL)
    if near: win_count[near[0]] += 1

deg = Counter()
for a, b in edges: deg[a] += 1; deg[b] += 1
isolated = [i for i in range(N) if deg[i] == 0]

# Adjacency fallback: a balcony is accessed from the room it adjoins, but if its
# access door isn't in aperture.obj it would be left isolated. Connect every still-
# isolated room to its nearest room (by centroid distance) so the graph stays connected.
ADJ_FALLBACK_TOL = 8.0
_c2 = cen_xyz[:, :2]
_extra = []
for i in isolated:
    d = ((_c2 - _c2[i])**2).sum(1); d[i] = 1e18
    j = int(d.argmin())
    if d[j]**0.5 <= ADJ_FALLBACK_TOL:
        _extra.append(tuple(sorted((i, j))))
if _extra:
    edge_set |= set(_extra); edges = sorted(edge_set)
    deg = Counter()
    for a, b in edges: deg[a] += 1; deg[b] += 1
    isolated = [i for i in range(N) if deg[i] == 0]
    print(f"adjacency fallback: +{len(set(_extra))} edge(s); isolated now {len(isolated)}")
print(f"circulation edges: {len(edges)}   doors not bridging 2 rooms: {unmatched_doors}")
print(f"connected rooms: {N - len(isolated)}/{N}   isolated: {len(isolated)}")
et = Counter(tuple(sorted((rtypes[a], rtypes[b]))) for a, b in edges)
print("\nTop adjacency types:")
for pair, c in et.most_common(10):
    print(f"  {pair[0]:11s} - {pair[1]:11s}: {c}")

doors: 106   windows: 22
adjacency fallback: +6 edge(s); isolated now 0
circulation edges: 102   doors not bridging 2 rooms: 6
connected rooms: 101/101   isolated: 0

Top adjacency types:
  bedroom     - corridor   : 19
  corridor    - livingroom : 18
  bathroom    - corridor   : 17
  corridor    - storeroom  : 10
  balcony     - livingroom : 9
  corridor    - corridor   : 6
  bathroom    - bedroom    : 5
  kitchen     - livingroom : 4
  bathroom    - livingroom : 3
  corridor    - kitchen    : 3


## 6. Visualise the semantic room-circulation graph (plan view)

In [6]:
fig = go.Figure()
# edges
ex, ey = [], []
for a, b in edges:
    ex += [cen_xyz[a,0], cen_xyz[b,0], None]
    ey += [cen_xyz[a,1], cen_xyz[b,1], None]
fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
                         line=dict(color="rgba(80,80,80,0.5)", width=1.2),
                         hoverinfo="skip", showlegend=False))
# nodes by room type
for t in present:
    idx = [i for i in range(N) if rtypes[i] == t]
    fig.add_trace(go.Scatter(
        x=cen_xyz[idx,0], y=cen_xyz[idx,1], mode="markers",
        marker=dict(size=12, color=ROOM_COLOR[t], line=dict(color="black", width=1)),
        name=f"{t} ({len(idx)})",
        text=[f"{t} (deg {deg[i]}, win {win_count[i]})" for i in idx], hoverinfo="text"))
fig.update_layout(
    title=dict(text="Floor 03 — semantic room-circulation graph", font=dict(color="black")),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(color="black", showgrid=True, gridcolor="rgba(0,0,0,0.08)", scaleanchor="y", scaleratio=1),
    yaxis=dict(color="black", showgrid=True, gridcolor="rgba(0,0,0,0.08)"),
    legend=dict(font=dict(color="black")), width=1100, height=750)
fig.show(renderer=renderer)

## 7. Spatial metrics on the room graph (bonus)

Quick space-syntax style metrics on the room graph: degree (most-connected rooms), closeness
(integration) and betweenness (choice). Computed with `topologicpy.Graph`.

In [7]:
gverts = [Vertex.ByCoordinates(float(cen_xyz[i,0]), float(cen_xyz[i,1]), float(cen_xyz[i,2])) for i in range(N)]
gedges = [Edge.ByVertices([gverts[a], gverts[b]]) for a, b in edges]
room_graph = Graph.ByVerticesEdges(gverts, gedges)
print(f"Room graph: {len(Graph.Vertices(room_graph))} nodes, {len(Graph.Edges(room_graph))} edges, "
      f"density {Graph.Density(room_graph):.4f}")

try:
    dc = Graph.DegreeCentrality(room_graph, normalize=True)
    cc = Graph.ClosenessCentrality(room_graph)
    bc = Graph.BetweennessCentrality(room_graph, normalize=True)
    for name, vals in [("Degree", dc), ("Closeness", cc), ("Betweenness", bc)]:
        order = np.argsort(vals)[::-1][:5]
        top = ", ".join(f"{rtypes[i]}({vals[i]:.3f})" for i in order)
        print(f"  Top-5 {name:11s}: {top}")
except Exception as e:
    print("centrality skipped:", e)

Room graph: 101 nodes, 102 edges, density 0.0202
  Top-5 Degree     : corridor(0.080), corridor(0.080), corridor(0.080), corridor(0.070), corridor(0.070)
  Top-5 Closeness  : corridor(0.247), corridor(0.223), corridor(0.219), corridor(0.215), livingroom(0.214)
  Top-5 Betweenness: corridor(0.414), corridor(0.297), corridor(0.283), corridor(0.274), livingroom(0.208)


## 8. Export the dataset in MSD CSV schema

Writes `nodes.csv`, `edges.csv`, `graphs.csv` to `dataset_F03/`, matching the schema that
`PyG.ByCSVPath` reads. Node features = `zoning` (4-d) + `connectivity` (3-d); edge features =
door connectivity (3-d). A stratified **train/val/test** node split is written into the masks.

In [8]:
# stratified per-class split: 3 train / 1 val / 1 test, round-robin within each class
split = [""] * N
cnt = Counter()
order = list(range(N)); random.shuffle(order)
for i in order:
    cnt[rtypes[i]] += 1
    k = cnt[rtypes[i]] % 5
    split[i] = "train" if k < 3 else ("val" if k == 3 else "test")

node_rows = []
for i in range(N):
    z = ZONING[rtypes[i]]; c = NODE_CONN[rtypes[i]]
    node_rows.append(dict(
        graph_id=0, node_id=i, label=LOCAL[rtypes[i]],
        feat_zoning_type_0=z[0], feat_zoning_type_1=z[1], feat_zoning_type_2=z[2], feat_zoning_type_3=z[3],
        feat_connectivity_0=c[0], feat_connectivity_1=c[1], feat_connectivity_2=c[2],
        train_mask=int(split[i]=="train"), val_mask=int(split[i]=="val"), test_mask=int(split[i]=="test")))
pd.DataFrame(node_rows).to_csv(DATASET_DIR / "nodes.csv", index=False)

edge_rows = []
for a, b in edges:
    for s, d in [(a, b), (b, a)]:   # directed both ways
        edge_rows.append(dict(graph_id=0, src_id=s, dst_id=d,
            feat_connectivity_0=DOOR_CONN[0], feat_connectivity_1=DOOR_CONN[1], feat_connectivity_2=DOOR_CONN[2]))
pd.DataFrame(edge_rows).to_csv(DATASET_DIR / "edges.csv", index=False)
pd.DataFrame([dict(graph_id=0, num_nodes=N)]).to_csv(DATASET_DIR / "graphs.csv", index=False)

print("Wrote:", DATASET_DIR / "nodes.csv", "/ edges.csv / graphs.csv")
print("Split:", dict(Counter(split)), "| classes:", N_CLASS)
print(pd.DataFrame(node_rows).head())

Wrote: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03\nodes.csv / edges.csv / graphs.csv
Split: {'test': 20, 'train': 60, 'val': 21} | classes: 8
   graph_id  node_id  label  feat_zoning_type_0  feat_zoning_type_1  \
0         0        0      0                   1                   0   
1         0        1      0                   1                   0   
2         0        2      0                   1                   0   
3         0        3      0                   1                   0   
4         0        4      0                   1                   0   

   feat_zoning_type_2  feat_zoning_type_3  feat_connectivity_0  \
0                   0                   0                    0   
1                   0                   0                    0   
2                   0                   0                    0   
3                   0                   0                    0   
4                   0                   0                    0   

   fea

## 9. Load the dataset into PyTorch-Geometric

In [9]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_DIR), level="node", task="classification",
    graphLabelType="categorical", nodeLabelType="categorical", edgeLabelType="categorical")
s = pyg.Summary()
print(f"Loaded: {s['num_graphs']} graph(s), {s['num_outputs']} classes, conv={s['conv']}, "
      f"hidden={s['hidden_dims']}")

Loaded: 1 graph(s), 8 classes, conv=sage, hidden=(64, 64)


## 10. Train the GNN node classifier

A GraphSAGE network predicts each room's type from its features + circulation neighbourhood.
Trained on the `train` nodes, early-stopped on `val`.

In [10]:
pyg.SetHyperparameters(epochs=200, lr=0.01, dropout=0.2,
                       early_stopping=True, early_stopping_patience=30)
t0 = time.time()
history = pyg.Train()
print(f"Trained in {time.time()-t0:.1f}s")
try:
    pyg.PlotHistory()
except Exception as e:
    print("PlotHistory skipped:", e)

Trained in 1.9s


## 11. Evaluate on the held-out test rooms

In [11]:
metrics = pyg.Test()
print("Test metrics:")
for k, v in metrics.items():
    print(f"  {k:16s}: {v:.4f}")
try:
    pyg.PlotConfusionMatrix(split="test")
except Exception as e:
    print("Confusion matrix skipped:", e)

Test metrics:
  test_accuracy   : 0.9500
  test_precision  : 0.9583
  test_recall     : 0.9500
  test_f1         : 0.9439


## 12. Predict room types for every room + export `node_predictions.csv`

In [12]:
report = pyg.Predict(split="all", return_probs=True)
pred = report["pred"]; true = report.get("y_true")

def to_class(val):
    a = np.squeeze(np.asarray(val))
    if a.ndim == 0: return int(a)
    return int(np.argmax(a)) if a.size > 1 else int(a[0])

# pred/true come back per-graph (one graph here)
gp = np.asarray(pred[0]) if isinstance(pred, (list, tuple)) else np.asarray(pred)
gt = np.asarray(true[0]) if (true is not None and isinstance(true, (list, tuple))) else None

rows = []
for i in range(N):
    yp = to_class(gp[i])
    yt = LOCAL[rtypes[i]]
    rows.append(dict(node_id=i, room_type=rtypes[i],
                     y_true=yt, y_pred=yp,
                     true_name=NAME[yt], pred_name=NAME.get(yp, "?"),
                     split=split[i], x=round(float(cen_xyz[i,0]),2), y=round(float(cen_xyz[i,1]),2)))
pred_df = pd.DataFrame(rows)
pred_df.to_csv(DATASET_DIR / "node_predictions.csv", index=False)

acc_all = (pred_df.y_true == pred_df.y_pred).mean()
acc_test = (pred_df[pred_df.split=="test"].y_true == pred_df[pred_df.split=="test"].y_pred).mean()
print(f"Accuracy — all rooms: {acc_all:.3f}   |   test rooms: {acc_test:.3f}")
print("\nMisclassified rooms:")
print(pred_df[pred_df.y_true != pred_df.y_pred][["node_id","true_name","pred_name","split"]].to_string(index=False))

Accuracy — all rooms: 0.931   |   test rooms: 0.950

Misclassified rooms:
 node_id true_name pred_name split
      83 storeroom  bathroom train
      84 storeroom  bathroom train
      85 storeroom  bathroom   val
      86 storeroom  bathroom train
      89 storeroom  bathroom  test
      90 storeroom  bathroom train
      91 storeroom  bathroom   val


## 13. Visualise true vs predicted room types (plan view)

In [13]:
fig = go.Figure()
ex, ey = [], []
for a, b in edges:
    ex += [cen_xyz[a,0], cen_xyz[b,0], None]; ey += [cen_xyz[a,1], cen_xyz[b,1], None]

def panel(col, name_key, title, xref):
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines", xaxis=xref,
                             line=dict(color="rgba(80,80,80,0.4)", width=1), hoverinfo="skip", showlegend=False))
    for t in present:
        idx = [i for i in range(N) if pred_df.iloc[i][name_key] == t]
        if not idx: continue
        wrong = [i for i in idx if pred_df.iloc[i].y_true != pred_df.iloc[i].y_pred]
        fig.add_trace(go.Scatter(
            x=cen_xyz[idx,0], y=cen_xyz[idx,1], mode="markers", xaxis=xref,
            marker=dict(size=12, color=ROOM_COLOR[t],
                        line=dict(color=["red" if i in wrong else "black" for i in idx],
                                  width=[3 if i in wrong else 1 for i in idx])),
            name=t, showlegend=(xref=="x"),
            text=[f"{pred_df.iloc[i].true_name}->{pred_df.iloc[i].pred_name}" for i in idx], hoverinfo="text"))

panel(None, "true_name", "True", "x")
panel(None, "pred_name", "Predicted", "x2")
fig.update_layout(
    title=dict(text="Room types — TRUE (left) vs PREDICTED (right); red ring = misclassified",
               font=dict(color="black")),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(domain=[0,0.48], color="black", scaleanchor="y", scaleratio=1, title="TRUE"),
    xaxis2=dict(domain=[0.52,1.0], color="black", scaleanchor="y", scaleratio=1, title="PREDICTED"),
    yaxis=dict(color="black"), legend=dict(font=dict(color="black")),
    width=1300, height=650)
fig.show(renderer=renderer)

## 14. Summary

- Rooms loaded as typed closed **cells** → graph **nodes**.
- **Doors** matched by proximity → **circulation edges** (handles real wall thickness).
- Dataset exported in **MSD CSV schema** → trained a **GraphSAGE** node classifier.
- Predicted each room's type and compared to the true labels.

**Next steps:** add Floor 04 the same way and join the floors through the `stairs`/`core` to get
a multi-floor semantic graph; optionally cross-validate (`pyg.CrossValidate(...)`) for a more
robust accuracy estimate, or swap in the course's pretrained MSD model via `pyg.LoadModel(...)`.

In [14]:
cv_note = "Run pyg.CrossValidate(k_folds=5, epochs=200) for a k-fold accuracy estimate (more reliable on small data)."
print(cv_note)
print(f"\nArtifacts in: {DATASET_DIR}")
for f in ["nodes.csv","edges.csv","graphs.csv","node_predictions.csv"]:
    p = DATASET_DIR / f
    print(f"  {'OK ' if p.exists() else 'MISSING '}{f}")

Run pyg.CrossValidate(k_folds=5, epochs=200) for a k-fold accuracy estimate (more reliable on small data).

Artifacts in: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03
  OK nodes.csv
  OK edges.csv
  OK graphs.csv
  OK node_predictions.csv


## 15. Export findings (summary + metadata + figure)

Collects every result above into a human-readable **`analysis_summary.md`**, a machine-readable
**`analysis_metadata.json`**, and a one-page **`15_summary.png`** (room-type distribution +
confusion matrix), all written to `dataset_F03/`. Uses `globals()` guards so it is safe to run
after a partial execution.

In [15]:
import json as _json
from datetime import datetime
from plotly.subplots import make_subplots
from collections import Counter as _C

G = globals()
meta = {}
meta["generated"]   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
meta["notebook"]    = "DubaiTower_F03_Semantic_Prediction.ipynb"
meta["description"] = "Kifaf Towers - Floor 03 semantic room graph + GNN room-type prediction."

meta["parameters"] = {
    "FLOOR_TAG": FLOOR_TAG,
    "room_types": list(ROOM_FILES.keys()),
    "proximity_tol": PROXIMITY_TOL,
    "n_classes": N_CLASS,
    "model": "GraphSAGE (topologicpy.PyG)",
}

dist = dict(_C(rtypes))
meta["graph"] = {
    "rooms": N, "edges": len(edges), "isolated": len(isolated),
    "density": round(float(Graph.Density(room_graph)), 5) if "room_graph" in G else None,
    "distribution": dist,
}
meta["adjacency_top"] = [{"pair": list(p), "count": c}
                         for p, c in _C(tuple(sorted((rtypes[a], rtypes[b]))) for a, b in edges).most_common(12)]

def _top5(vals):
    order = np.argsort(vals)[::-1][:5]
    return [{"room": rtypes[i], "score": round(float(vals[i]), 4)} for i in order]
meta["centrality_top5"] = {}
for nm, key in [("dc", "degree"), ("cc", "closeness"), ("bc", "betweenness")]:
    if nm in G:
        meta["centrality_top5"][key] = _top5(G[nm])

if "metrics" in G:
    meta["test_metrics"] = {k: round(float(v), 4) for k, v in metrics.items()}

if "pred_df" in G:
    per_class = {}
    for t in present:
        sub = pred_df[pred_df.true_name == t]
        per_class[t] = round(float((sub.y_true == sub.y_pred).mean()), 3) if len(sub) else None
    meta["per_class_accuracy"] = per_class
    meta["accuracy_all"]  = round(float((pred_df.y_true == pred_df.y_pred).mean()), 4)
    meta["accuracy_test"] = round(float((pred_df[pred_df.split=="test"].y_true ==
                                         pred_df[pred_df.split=="test"].y_pred).mean()), 4)
    meta["misclassified"] = [
        {"node": int(r.node_id), "true": r.true_name, "pred": r.pred_name, "split": r.split}
        for _, r in pred_df[pred_df.y_true != pred_df.y_pred].iterrows()]

json_path = DATASET_DIR / "analysis_metadata.json"
with open(json_path, "w", encoding="utf-8") as f:
    _json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved:", json_path)

L = []
L.append("# Analysis Summary - Kifaf Towers Floor 03 (Semantic Room Graph + GNN)\n")
L.append(f"_Generated: {meta['generated']}_\n")
L.append(meta["description"] + "\n")
L.append("## Parameters\n")
L.append("| Parameter | Value |"); L.append("|---|---|")
for k, v in meta["parameters"].items():
    L.append(f"| `{k}` | {v} |")
L.append("")
g = meta["graph"]
L.append("## Room graph\n")
L.append(f"- Rooms (nodes): **{g['rooms']}**  |  Edges: **{g['edges']}**  |  "
         f"Isolated: {g['isolated']}  |  Density: {g['density']}\n")
L.append("| Room type | Count |"); L.append("|---|---|")
for t in present:
    L.append(f"| {t} | {g['distribution'].get(t, 0)} |")
L.append("")
L.append("## Top adjacency types\n")
L.append("| Pair | Count |"); L.append("|---|---|")
for a in meta["adjacency_top"]:
    L.append(f"| {a['pair'][0]} - {a['pair'][1]} | {a['count']} |")
L.append("")
if meta.get("centrality_top5"):
    L.append("## Centrality (top-5 rooms)\n")
    for key, lst in meta["centrality_top5"].items():
        L.append(f"- **{key.capitalize()}**: " + ", ".join(f"{d['room']} ({d['score']})" for d in lst))
    L.append("")
if "test_metrics" in meta:
    L.append("## Node classification - test metrics\n")
    L.append("| Metric | Value |"); L.append("|---|---|")
    for k, v in meta["test_metrics"].items():
        L.append(f"| {k} | {v} |")
    L.append(f"\n- Accuracy (all rooms): **{meta.get('accuracy_all')}**  |  "
             f"Accuracy (test rooms): **{meta.get('accuracy_test')}**\n")
if "per_class_accuracy" in meta:
    L.append("## Per-class accuracy\n")
    L.append("| Room type | Accuracy |"); L.append("|---|---|")
    for t, a in meta["per_class_accuracy"].items():
        L.append(f"| {t} | {a} |")
    L.append("")
if meta.get("misclassified"):
    L.append("## Misclassified rooms\n")
    L.append("| Node | True | Pred | Split |"); L.append("|---|---|---|---|")
    for m in meta["misclassified"]:
        L.append(f"| {m['node']} | {m['true']} | {m['pred']} | {m['split']} |")
    L.append("")
L.append("---")
L.append("_Generated automatically by section 15 of the notebook._")

md_path = DATASET_DIR / "analysis_summary.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(L))
print("Saved:", md_path)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                    subplot_titles=["Room-type distribution", "Room-type confusion matrix"],
                    column_widths=[0.42, 0.58])
fig.add_trace(go.Bar(x=present, y=[dist.get(t, 0) for t in present],
                     marker_color=[ROOM_COLOR[t] for t in present],
                     marker_line=dict(color="black", width=1), showlegend=False), row=1, col=1)
if "pred_df" in G:
    cm = np.zeros((N_CLASS, N_CLASS), int)
    for _, r in pred_df.iterrows():
        cm[int(r.y_true), int(r.y_pred)] += 1
    names = [NAME[i] for i in range(N_CLASS)]
    fig.add_trace(go.Heatmap(z=cm, x=names, y=names, colorscale="Blues",
                             text=cm, texttemplate="%{text}", showscale=True,
                             colorbar=dict(tickfont=dict(color="black"))), row=1, col=2)
    fig.update_xaxes(title_text="predicted", row=1, col=2, color="black")
    fig.update_yaxes(title_text="true", row=1, col=2, color="black")
acc = meta.get("accuracy_test")
fig.update_xaxes(color="black", row=1, col=1); fig.update_yaxes(color="black", row=1, col=1)
for ann in fig.layout.annotations: ann.font.color = "black"
fig.update_layout(title=dict(text=f"Floor 03 - GNN room-type prediction  (test accuracy {acc})",
                             font=dict(color="black")),
                  paper_bgcolor="white", plot_bgcolor="white", width=1300, height=560)
fig.show(renderer=renderer)
png_path = DATASET_DIR / "15_summary.png"
try:
    fig.write_image(str(png_path), width=1300, height=560, scale=2)
    print("Saved:", png_path)
except Exception as e:
    print("Could not save summary PNG (kaleido?):", e)

print("\n" + "\n".join(L[:16]))


Saved: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03\analysis_metadata.json
Saved: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03\analysis_summary.md


Saved: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_F03\15_summary.png

# Analysis Summary - Kifaf Towers Floor 03 (Semantic Room Graph + GNN)

_Generated: 2026-06-25 23:38:13_

Kifaf Towers - Floor 03 semantic room graph + GNN room-type prediction.

## Parameters

| Parameter | Value |
|---|---|
| `FLOOR_TAG` | F03 |
| `room_types` | ['bedroom', 'livingroom', 'kitchen', 'corridor', 'stairs', 'bathroom', 'storeroom', 'balcony'] |
| `proximity_tol` | 0.6 |
| `n_classes` | 8 |
| `model` | GraphSAGE (topologicpy.PyG) |

## Room graph

- Rooms (nodes): **101**  |  Edges: **102**  |  Isolated: 0  |  Density: 0.0202

| Room type | Count |
|---|---|
